# Pre-processing Allocation data for WaDE upload.
- Purpose:  To pre-process the data into one main file for simple DataFrame creation and extraction

In [1]:
import sys
print(sys.executable)

C:\Users\rjame\anaconda3\envs\wade_data_project\python.exe


In [2]:
import os
import sys
print(os.environ['CONDA_DEFAULT_ENV'])
print(sys.version)

wade_data_project
3.13.5 | packaged by conda-forge | (main, Jun 16 2025, 08:20:19) [MSC v.1943 64 bit (AMD64)]


In [3]:
# Needed Libraries / Modules

# ---- working with data ----
import numpy as np  # mathematical array manipulation
import pandas as pd  # data structure and data analysis
import geopandas as gpd  # geo-data structure and data analysis

# ---- visualization ----
import matplotlib.pyplot as plt  # plotting library
import seaborn as sns  # plotting library

# ---- API data retrieval ----
import requests  # http requests
import json  # JSON parse

# ---- Cleanup ----
import re  # string regular expression manipulation
from datetime import datetime  # date and time manipulation
pd.set_option('display.max_columns', 999)  # How to display all columns of a Pandas DataFrame in Jupyter Notebook
pd.set_option('display.float_format', lambda x: '%.5f' % x)  # suppress scientific notation in Pandas

In [4]:
# ---- working directory ----
workingDirString = "G:/Shared drives/WaDE Data/WaDE Data Folder/Kansas/WaterAllocation" # set working directory folder string here
os.chdir(workingDirString)
print(f'The working Directory is:', workingDirString)

The working Directory is: G:/Shared drives/WaDE Data/WaDE Data Folder/Kansas/WaterAllocation


## Point of Diversion Data

In [5]:
# Input File
fileInput = "RawInputData/qty_20260205502.zip"
df_qty = pd.read_csv(fileInput, compression='zip')
df_qty = df_qty[['wr_id', 'pdiv_id', 'auth_quant', 'quant_unit']]
df_qty['KeyJoin'] = df_qty['wr_id'].astype(str) + "_" + df_qty['pdiv_id'].astype(str)
print(len(df_qty))
df_qty.head(1)

47153


C:\Users\rjame\AppData\Local\Temp\ipykernel_5460\111667335.py:3: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df_qty = pd.read_csv(fileInput, compression='zip')


,wr_id,pdiv_id,auth_quant,quant_unit,KeyJoin
0,24738,128,203,AF,24738_128


In [6]:
# Input File
fileInput = "RawInputData/wimas_20260205600.zip"
df_wimas = pd.read_csv(fileInput, compression='zip')
df_wimas['KeyJoin'] = df_wimas['wr_id'].astype(str) + "_" + df_wimas['pdiv_id'].astype(str)
print(len(df_wimas))
df_wimas.head(1)

89150


,wr_id,right_type,vested_county_code,wr_num,wr_qualifier,umw_code,wrfile_active_ind,source_of_supply,current_status_code,priority_date,pdiv_id,fpdiv_active_ind,township,township_dir,range_num,range_dir,section_num,dwr_id,qual1,qual2,qual3,qual4,longitude,latitude,fpdiv_comment,feet_north,feet_west,basin_num,gmd,fo_num,county_code,stream_num,num_wells,lot_number,lot_qualifier_one,lot_qualifier_two,well_kid,wimas_date,WaDEUUID,KeyJoin
0,110,A,,109,00,IRR,0,S,NQ,20-MAY-1946,46914,1,7,S,12,W,22,1,SE,,,,-98.64340,39.42701,NaN,NaN,,25,,3,OB,25,,,,,,02/01/2026,d0,110_46914


In [7]:
# Input File
dfinPOD = pd.merge(df_qty, df_wimas, left_on='KeyJoin', right_on='KeyJoin', how='inner')
dfinPOD = dfinPOD.drop_duplicates().reset_index(drop=True)

# WaDE UUID tracker for data assessment
if 'WaDEUUID' not in dfinPOD:
    dfinPOD['WaDEUUID'] = "d" + dfinPOD.index.astype(str)
    dfinPOD.to_csv('RawInputData/wimas_qyt_20260205600.zip', compression=dict(method='zip', archive_name='wimas_qyt_20260205600.csv'), index=False)

print(len(dfinPOD))
dfinPOD.head()

48854


,wr_id_x,pdiv_id_x,auth_quant,quant_unit,KeyJoin,wr_id_y,right_type,vested_county_code,wr_num,wr_qualifier,umw_code,wrfile_active_ind,source_of_supply,current_status_code,priority_date,pdiv_id_y,fpdiv_active_ind,township,township_dir,range_num,range_dir,section_num,dwr_id,qual1,qual2,qual3,qual4,longitude,latitude,fpdiv_comment,feet_north,feet_west,basin_num,gmd,fo_num,county_code,stream_num,num_wells,lot_number,lot_qualifier_one,lot_qualifier_two,well_kid,wimas_date,WaDEUUID
0,24738,128,203,AF,24738_128,24738,A,,24317,00,IRR,1,G,NK,28-JUL-1975,128,1,4,S,33,W,35,1,NW,SW,SW,,-100.99873,39.66212,NaN,2710.00000,4820,29,4,3,RA,,1,,,,1040853725,02/01/2026,d35260
1,34638,749,260,AF,34638_749,34638,A,,34112,00,IRR,1,G,NK,15-APR-1980,749,1,27,S,34,W,7,1,SE,NC,,,-101.07568,37.71151,NaN,1450.00000,1170,33,3,4,HS,,1,,,,1040166072,02/01/2026,d52784
2,36303,753,59.44,AF,36303_753,36303,A,,35768,00,IRR,1,G,NK,25-NOV-1981,753,1,8,S,23,W,20,1,NW,NE,NE,,-99.91405,39.34800,NaN,4720.00000,3200,25,,3,GH,,1,,,,1040160789,02/01/2026,d57232
3,86945,89686,224,AF,86945_89686,86945,A,,50395,00,IRR,1,G,HK,11-MAY-2020,89686,1,31,S,8,W,10,10,NE,NW,SE,,-98.17381,37.36612,BATT 1 OF 2 WELLS,3971.00000,1945,49,,2,HP,,2,,,,1052689905,02/01/2026,d86128
4,43091,1676,180,AF,43091_1676,43091,A,,42441,00,IRR,1,G,NK,29-AUG-1996,1676,1,26,S,6,W,9,2,NW,SW,SW,,-97.99142,37.79939,BATT 1 OF 2 WELLS,2688.00000,5258,53,2,2,RN,,2,,,,1040391679,02/01/2026,d63061


In [8]:
#Right Type Code
rightTypeDict = {
"A" : "Appropriation",
"B" : "Basin Term",
"D" : "Domestic",
"P" : "Temporary",
"T" : "Term",
"V" : "Vested"
}

def retrieveRightType(colrowValue):
    if colrowValue == "" or pd.isnull(colrowValue):
        outList = ""
    else:
        String1 = str(colrowValue).strip()
        try:
            outList = rightTypeDict[String1]
        except:
            outList = ""
    return outList

dfinPOD['in_AllocationTypeCV'] = dfinPOD.apply(lambda row: retrieveRightType(row['right_type']), axis=1)
dfinPOD['in_AllocationTypeCV'].unique()

array(['Appropriation', 'Vested'], dtype=object)

In [9]:
#BenUse Code
useTypeDict = {
"ART" : "Artificial Recharge",
"CON" : "Contamination Remediation",
"DEW" : "Dewatering",
"DOM" : "Domestic",
"FPR" : "Fire Protection",
"HYD" : "Hydraulic Dredging",
"IND" : "Industrial",
"IRR" : "Irrigation",
"MUN" : "Municipal",
"REC" : "Recreation",
"SED" : "Sediment Storage",
"STK" : "Stockwater",
"THX" : "Thermal Exchange",
"WTR" : "Water Power"
}

def retrieveUseType(colrowValue):
    if colrowValue == "" or pd.isnull(colrowValue):
        outList = ""
    else:
        String1 = str(colrowValue).strip()
        try:
            outList = useTypeDict[String1]
        except:
            outList = ""
    return outList

dfinPOD['in_BeneficialUseCategory'] = dfinPOD.apply(lambda row: retrieveUseType(row['umw_code']), axis=1)
dfinPOD['in_BeneficialUseCategory'].unique()

array(['Irrigation', 'Municipal', 'Industrial', 'Stockwater', 'Domestic',
       'Recreation'], dtype=object)

In [10]:
#Watersource Type Code
wsTypeDict = {
"S" : "Surface Water",
"G" : "Groundwater"}

def retrieveWSType(colrowValue):
    if colrowValue == "" or pd.isnull(colrowValue):
        outList = ""
    else:
        String1 = str(colrowValue).strip()
        try:
            outList = wsTypeDict[String1]
        except:
            outList = ""
    return outList

dfinPOD['in_WatersourceType'] = dfinPOD.apply(lambda row: retrieveWSType(row['source_of_supply']), axis=1)
dfinPOD['in_WatersourceType'].unique()

array(['Groundwater', 'Surface Water'], dtype=object)

In [11]:
#Status Type Code
statusTypeDict = {
"AA" : "Vested Active",
"AM" : "Dismissed After Vested",
"AY" : "Pending Initial Review",
"FO" : "Dismissed Prior to Approval",
"GA" : "Denied Prior to approval",
"GM" : "Reinstated Prior to Approval",
"GY" : "Approved Pending Completion",
"HK" : "Extended Time to Complete",
"HW" : "Dismissed Pending Completion",
"II" : "Reinstated Pending Completion",
"IU" : "Partial Completion",
"JG" : "Partial Completion Extended Time to Complete",
"JM" : "Inspected Prior to Completion",
"KE" : "Completed Pending Inspection",
"KK" : "Completed Extended Time to Perfect",
"KQ" : "Dismissed Pending Inspection",
"LC" : "Reinstated Pending inspection",
"LG" : "Completed Partial inspection",
"LK" : "Partial Inspection Extended Time to Perfect",
"LO" : "Inspected Pending Perfection",
"LR" : "Inspected Pending Perfection Extended Time to Perfect",
"LU" : "Dismissed Pending Perfection",
"LZ" : "Reinstated Pending Perfection",
"MM" : "Proposed Certificate",
"MR" : "Proposed Certificate Extended Time to Perfect",
"NK" : "Certificated Issued",
"NQ" : "Dismissed After Certificated Issued",
"NT" : "Reinstated After Certificate Issued",
"NV" : "Reinstated After Vested"
}

def retrieveStatusType(colrowValue):
    if colrowValue == "" or pd.isnull(colrowValue):
        outList = ""
    else:
        String1 = str(colrowValue).strip()
        try:
            outList = statusTypeDict[String1]
        except:
            outList = ""
    return outList

dfinPOD['in_AllocationLegalStatusCV'] = dfinPOD.apply(lambda row: retrieveStatusType(row['current_status_code']), axis=1)
dfinPOD['in_AllocationLegalStatusCV'].unique()

array(['Certificated Issued', 'Extended Time to Complete',
       'Vested Active', 'Dismissed Pending Completion',
       'Completed Pending Inspection',
       'Inspected Pending Perfection Extended Time to Perfect',
       'Inspected Pending Perfection', 'Approved Pending Completion',
       'Proposed Certificate', 'Dismissed After Certificated Issued',
       'Completed Extended Time to Perfect',
       'Dismissed Pending Perfection', '', 'Dismissed Pending Inspection',
       'Proposed Certificate Extended Time to Perfect',
       'Dismissed Prior to Approval', 'Dismissed After Vested',
       'Partial Inspection Extended Time to Perfect',
       'Reinstated After Certificate Issued',
       'Completed Partial inspection', 'Reinstated After Vested'],
      dtype=object)

In [12]:
#Basin Code
basinDict = {
"1" : "Missouri River",
"2" : "S F Big Nemaha River",
"3" : "Marais Des Cygnes River",
"4" : "Sugar Creek",
"5" : "Pottawatomie Creek",
"6" : "Little Osage River",
"7" : "Marmaton River",
"8" : "Kansas River",
"9" : "Stranger Creek",
"10" : "Wakarusa River",
"11" : "Delaware River",
"12" : "Vermillion Creek",
"13" : "Big Blue River",
"14" : "Black Vermillion River",
"15" : "Little Blue River",
"16" : "Mill Creek",
"17" : "Smoky Hill River",
"18" : "Saline River",
"19" : "Big Creek",
"20" : "Hackberry Creek",
"21" : "Ladder Creek",
"22" : "N F Smoky Hill River",
"23" : "Solomon River",
"24" : "Salt Creek",
"25" : "S F Solomon River",
"26" : "N F Solomon River",
"27" : "Republican River",
"28" : "Prairie Dog Creek",
"29" : "Sappa Creek",
"30" : "Beaver Creek",
"31" : "S F Republican River",
"32" : "Arikaree River",
"33" : "Arkansas River",
"34" : "Neosho River",
"35" : "Spring River",
"36" : "Cottonwood River",
"37" : "Verdigris River",
"38" : "Caney River",
"39" : "Elk River",
"40" : "Fall River",
"41" : "Cimarron River",
"42" : "Bluff Creek (cimarron)",
"43" : "Crooked Creek",
"44" : "N F Cimarron River",
"45" : "Bear Creek",
"46" : "Salt Fork Arkansas River",
"47" : "Medicine Lodge River",
"48" : "Chikaskia River",
"49" : "Bluff Creek (chikaskia)",
"50" : "Sandy Creek",
"51" : "Walnut River",
"52" : "Ninnescah River",
"53" : "N F Ninnescah River",
"54" : "S F Ninnescah River",
"55" : "Little Arkansas River",
"56" : "Cow Creek",
"57" : "Rattlesnake Creek",
"58" : "Walnut Creek",
"59" : "Pawnee River",
"60" : "Buckner Creek",
"61" : "Whitewoman Creek",
"62" : "Driftwood Creek"
}

def retrieveBasin(colrowValue):
    if colrowValue == "" or pd.isnull(colrowValue):
        outList = ""
    else:
        String1 = str(colrowValue).strip()
        try:
            outList = basinDict[String1]
        except:
            outList = ""
    return outList

dfinPOD['in_WaterSourceName'] = dfinPOD.apply(lambda row: retrieveBasin(row['basin_num']), axis=1)
dfinPOD['in_WaterSourceName'].unique()

array(['Sappa Creek', 'Arkansas River', 'S F Solomon River',
       'Bluff Creek (chikaskia)', 'N F Ninnescah River', 'Walnut Creek',
       'Whitewoman Creek', 'S F Ninnescah River', 'N F Cimarron River',
       'Little Arkansas River', 'Hackberry Creek', 'Solomon River',
       'Rattlesnake Creek', 'Missouri River', 'Republican River',
       'Chikaskia River', 'Bear Creek', 'Ladder Creek',
       'Smoky Hill River', 'N F Solomon River', 'Kansas River',
       'Big Blue River', 'Crooked Creek', 'Cimarron River',
       'Walnut River', 'Saline River', 'Medicine Lodge River',
       'Beaver Creek', 'Pawnee River', 'Prairie Dog Creek',
       'S F Republican River', 'Cow Creek', 'Big Creek', 'Buckner Creek',
       'Spring River', 'Ninnescah River', 'Sandy Creek',
       'Cottonwood River', 'Delaware River', 'Stranger Creek',
       'Little Blue River', 'Salt Creek', 'S F Big Nemaha River',
       'Marais Des Cygnes River', 'N F Smoky Hill River',
       'Bluff Creek (cimarron)', 'Wakar

In [13]:
#County Code
countyDict = {
"AL" : "Allen",
"AN" : "Anderson",
"AT" : "Atchison",
"BA" : "Barber",
"BT" : "Barton",
"BB" : "Bourbon",
"BR" : "Brown",
"BU" : "Butler",
"CS" : "Chase",
"CQ" : "Chautauqua",
"CK" : "Cherokee",
"CN" : "Cheyenne",
"CA" : "Clark",
"CY" : "Clay",
"CD" : "Cloud",
"CF" : "Coffey",
"CM" : "Comanche",
"CL" : "Cowley",
"CR" : "Crawford",
"DC" : "Decatur",
"DK" : "Dickinson",
"DP" : "Doniphan",
"DG" : "Douglas",
"ED" : "Edwards",
"EK" : "Elk",
"EL" : "Ellis",
"EW" : "Ellsworth",
"FI" : "Finney",
"FO" : "Ford",
"FR" : "Franklin",
"GE" : "Geary",
"GO" : "Gove",
"GH" : "Graham",
"GT" : "Grant",
"GY" : "Gray",
"GL" : "Greeley",
"GW" : "Greenwood",
"HM" : "Hamilton",
"HP" : "Harper",
"HV" : "Harvey",
"HS" : "Haskell",
"HG" : "Hodgeman",
"JA" : "Jackson",
"JF" : "Jefferson",
"JW" : "Jewell",
"JO" : "Johnson",
"KE" : "Kearny",
"KM" : "Kingman",
"KW" : "Kiowa",
"LB" : "Labette",
"LE" : "Lane",
"LV" : "Leavenworth",
"LC" : "Lincoln",
"LN" : "Linn",
"LG" : "Logan",
"LY" : "Lyon",
"MN" : "Marion",
"MS" : "Marshall",
"MP" : "McPherson",
"ME" : "Meade",
"MI" : "Miami",
"MC" : "Mitchell",
"MG" : "Montgomery",
"MR" : "Morris",
"MT" : "Morton",
"NM" : "Nemaha",
"NO" : "Neosho",
"NS" : "Ness",
"NT" : "Norton",
"OS" : "Osage",
"OB" : "Osborne",
"OT" : "Ottawa",
"PN" : "Pawnee",
"PL" : "Phillips",
"PT" : "Pottawatomie",
"PR" : "Pratt",
"RA" : "Rawlins",
"RN" : "Reno",
"RP" : "Republic",
"RC" : "Rice",
"RL" : "Riley",
"RO" : "Rooks",
"RH" : "Rush",
"RS" : "Russell",
"SA" : "Saline",
"SC" : "Scott",
"SG" : "Sedgwick",
"SW" : "Seward",
"SN" : "Shawnee",
"SD" : "Sheridan",
"SH" : "Sherman",
"SM" : "Smith",
"SF" : "Stafford",
"ST" : "Stanton",
"SV" : "Stevens",
"SU" : "Sumner",
"TH" : "Thomas",
"TR" : "Trego",
"WB" : "Wabaunsee",
"WA" : "Wallace",
"WS" : "Washington",
"WH" : "Wichita",
"WL" : "Wilson",
"WO" : "Woodson",
"WY" : "Wyandotte"}

def retrieveCounty(colrowValue):
    if colrowValue == "" or pd.isnull(colrowValue):
        outList = ""
    else:
        String1 = str(colrowValue).strip()
        try:
            outList = countyDict[String1]
        except:
            outList = ""
    return outList

dfinPOD['in_County'] = dfinPOD.apply(lambda row: retrieveCounty(row['county_code']), axis=1)
dfinPOD['in_County'].unique()

array(['Rawlins', 'Haskell', 'Graham', 'Harper', 'Reno', 'Pawnee', 'Lane',
       'Scott', 'Wichita', 'Kingman', 'Stanton', 'Grant', 'Kearny',
       'Harvey', 'Edwards', 'Logan', 'Mitchell', 'Sumner', 'Thomas',
       'Brown', 'Republic', 'Sedgwick', 'Hamilton', 'Ford', 'Trego',
       'Decatur', 'Pottawatomie', 'Gray', 'Pratt', 'Seward', 'Morris',
       'Cowley', 'Morton', 'Saline', 'Butler', 'Finney', 'Geary',
       'Sherman', 'Smith', 'Clay', 'Phillips', 'Wabaunsee', 'Wallace',
       'Ellis', 'Sheridan', 'McPherson', 'Rooks', 'Cheyenne', 'Meade',
       'Shawnee', 'Jefferson', 'Stevens', 'Rice', 'Kiowa', 'Gove', 'Rush',
       'Cloud', 'Barton', 'Crawford', 'Greeley', 'Stafford', 'Norton',
       'Ness', 'Russell', 'Douglas', 'Osborne', 'Barber', 'Ottawa',
       'Marion', 'Marshall', 'Washington', 'Lincoln', 'Riley', 'Hodgeman',
       'Cherokee', 'Coffey', 'Ellsworth', 'Dickinson', 'Leavenworth',
       'Comanche', 'Chase', 'Wyandotte', 'Johnson', 'Allen', 'Clark',
       'Jew

In [15]:
# create output POD dataframe
df = pd.DataFrame()

# Data Assessment UUID
df['WaDEUUID'] = dfinPOD['WaDEUUID']

# Method Info
df['in_MethodUUID'] = "KSwr_M1"

# Variable Info
df['in_VariableSpecificUUID'] = "KSwr_V1"

# Organization Info
df['in_OrganizationUUID'] = "KSwr_O1"

# WaterSource Info
df['in_Geometry'] = ""
df['in_GNISFeatureNameCV'] = ""
df['in_WaterQualityIndicatorCV'] = ""
df['in_WaterSourceName'] = dfinPOD['in_WaterSourceName']
df['in_WaterSourceNativeID'] = "" # auto fill in below
df['in_WaterSourceTypeCV'] = dfinPOD['in_WatersourceType']

# Site Info
df['in_CoordinateAccuracy'] = "WaDE Unspecified"
df['in_CoordinateMethodCV'] = "WaDE Unspecified"
df['in_County'] = dfinPOD['in_County']
df['in_EPSGCodeCV'] = 4326
df['in_Geometry'] = ""
df['in_GNISCodeCV'] = ""
df['in_HUC12'] = ""
df['in_HUC8'] = ""
df['in_Latitude'] = dfinPOD['latitude']
df['in_Longitude'] = dfinPOD['longitude']
df['in_NHDNetworkStatusCV'] = ""
df['in_NHDProductCV'] = ""
df['in_PODorPOUSite'] = "POD"
df['in_SiteName'] = "WaDE Unspecified"
df['in_SiteNativeID'] = "POD" + dfinPOD['pdiv_id_x'].replace("", 0).fillna(0).astype(int).astype(str)
df['in_SitePoint'] = ""
df['in_SiteTypeCV'] = "WaDE Unspecified"
df['in_StateCV'] = "KS"
df['in_USGSSiteID'] = ""

# AllocationAmount Info
df['in_AllocationApplicationDate'] = ""
df['in_AllocationAssociatedConsumptiveUseSiteIDs'] = ""
df['in_AllocationAssociatedWithdrawalSiteIDs'] = ""
df['in_AllocationBasisCV'] = ""
df['in_AllocationChangeApplicationIndicator'] = ""
df['in_AllocationCommunityWaterSupplySystem'] = ""
df['in_AllocationCropDutyAmount'] = ""
df['in_AllocationExpirationDate'] = ""
df['in_AllocationFlow_CFS'] = ""
df['in_AllocationLegalStatusCV'] = dfinPOD['in_AllocationLegalStatusCV']
df['in_AllocationNativeID'] =  "ks" + dfinPOD['wr_id_x'].replace("", 0).fillna(0).astype(str)
df['in_AllocationOwner'] = "WaDE Unspecified"
df['in_AllocationPriorityDate'] = dfinPOD['priority_date']
df['in_AllocationSDWISIdentifierCV'] = ""
df['in_AllocationTimeframeEnd'] = ""
df['in_AllocationTimeframeStart'] = ""
df['in_AllocationTypeCV'] = dfinPOD['in_AllocationTypeCV']
df['in_AllocationVolume_AF'] = dfinPOD['auth_quant']
df['in_BeneficialUseCategory'] = dfinPOD['in_BeneficialUseCategory']
df['in_CommunityWaterSupplySystem'] = ""
df['in_CropTypeCV'] = ""
df['in_CustomerTypeCV'] = ""
df['in_DataPublicationDate'] = ""
df['in_DataPublicationDOI'] = ""
df['in_ExemptOfVolumeFlowPriority'] = 0
df['in_GeneratedPowerCapacityMW'] = ""
df['in_IrrigatedAcreage'] = ""
df['in_IrrigationMethodCV'] = ""
df['in_LegacyAllocationIDs'] = ""
df['in_OwnerClassificationCV'] = ""
df['in_PopulationServed'] = ""
df['in_PowerType'] = ""
df['in_PrimaryBeneficialUseCategory'] = ""
df['in_SDWISIdentifierCV'] = ""
df['in_WaterAllocationNativeURL'] = "http://geohydro.kgs.ku.edu/geohydro/wimas/water_right_list_direct.cfm?wr_id=" + dfinPOD['wr_id_x'].replace("", 0).fillna(0).astype(int).astype(str)

outPOD = df.drop_duplicates().reset_index(drop=True)
print(len(outPOD))
outPOD.head()

48854


,WaDEUUID,in_MethodUUID,in_VariableSpecificUUID,in_OrganizationUUID,in_Geometry,in_GNISFeatureNameCV,in_WaterQualityIndicatorCV,in_WaterSourceName,in_WaterSourceNativeID,in_WaterSourceTypeCV,in_CoordinateAccuracy,in_CoordinateMethodCV,in_County,in_EPSGCodeCV,in_GNISCodeCV,in_HUC12,in_HUC8,in_Latitude,in_Longitude,in_NHDNetworkStatusCV,in_NHDProductCV,in_PODorPOUSite,in_SiteName,in_SiteNativeID,in_SitePoint,in_SiteTypeCV,in_StateCV,in_USGSSiteID,in_AllocationApplicationDate,in_AllocationAssociatedConsumptiveUseSiteIDs,in_AllocationAssociatedWithdrawalSiteIDs,in_AllocationBasisCV,in_AllocationChangeApplicationIndicator,in_AllocationCommunityWaterSupplySystem,in_AllocationCropDutyAmount,in_AllocationExpirationDate,in_AllocationFlow_CFS,in_AllocationLegalStatusCV,in_AllocationNativeID,in_AllocationOwner,in_AllocationPriorityDate,in_AllocationSDWISIdentifierCV,in_AllocationTimeframeEnd,in_AllocationTimeframeStart,in_AllocationTypeCV,in_AllocationVolume_AF,in_BeneficialUseCategory,in_CommunityWaterSupplySystem,in_CropTypeCV,in_CustomerTypeCV,in_DataPublicationDate,in_DataPublicationDOI,in_ExemptOfVolumeFlowPriority,in_GeneratedPowerCapacityMW,in_IrrigatedAcreage,in_IrrigationMethodCV,in_LegacyAllocationIDs,in_OwnerClassificationCV,in_PopulationServed,in_PowerType,in_PrimaryBeneficialUseCategory,in_SDWISIdentifierCV,in_WaterAllocationNativeURL
0,d35260,KSwr_M1,KSwr_V1,KSwr_O1,,,,Sappa Creek,,Groundwater,WaDE Unspecified,WaDE Unspecified,Rawlins,4326,,,,39.66212,-100.99873,,,POD,WaDE Unspecified,POD128,,WaDE Unspecified,KS,,,,,,,,,,,Certificated Issued,ks24738,WaDE Unspecified,28-JUL-1975,,,,Appropriation,203,Irrigation,,,,,,0,,,,,,,,,,http://geohydro.kgs.ku.edu/geohydro/wimas/wate...
1,d52784,KSwr_M1,KSwr_V1,KSwr_O1,,,,Arkansas River,,Groundwater,WaDE Unspecified,WaDE Unspecified,Haskell,4326,,,,37.71151,-101.07568,,,POD,WaDE Unspecified,POD749,,WaDE Unspecified,KS,,,,,,,,,,,Certificated Issued,ks34638,WaDE Unspecified,15-APR-1980,,,,Appropriation,260,Irrigation,,,,,,0,,,,,,,,,,http://geohydro.kgs.ku.edu/geohydro/wimas/wate...
2,d57232,KSwr_M1,KSwr_V1,KSwr_O1,,,,S F Solomon River,,Groundwater,WaDE Unspecified,WaDE Unspecified,Graham,4326,,,,39.34800,-99.91405,,,POD,WaDE Unspecified,POD753,,WaDE Unspecified,KS,,,,,,,,,,,Certificated Issued,ks36303,WaDE Unspecified,25-NOV-1981,,,,Appropriation,59.44,Irrigation,,,,,,0,,,,,,,,,,http://geohydro.kgs.ku.edu/geohydro/wimas/wate...
3,d86128,KSwr_M1,KSwr_V1,KSwr_O1,,,,Bluff Creek (chikaskia),,Groundwater,WaDE Unspecified,WaDE Unspecified,Harper,4326,,,,37.36612,-98.17381,,,POD,WaDE Unspecified,POD89686,,WaDE Unspecified,KS,,,,,,,,,,,Extended Time to Complete,ks86945,WaDE Unspecified,11-MAY-2020,,,,Appropriation,224,Irrigation,,,,,,0,,,,,,,,,,http://geohydro.kgs.ku.edu/geohydro/wimas/wate...
4,d63061,KSwr_M1,KSwr_V1,KSwr_O1,,,,N F Ninnescah River,,Groundwater,WaDE Unspecified,WaDE Unspecified,Reno,4326,,,,37.79939,-97.99142,,,POD,WaDE Unspecified,POD1676,,WaDE Unspecified,KS,,,,,,,,,,,Certificated Issued,ks43091,WaDE Unspecified,29-AUG-1996,,,,Appropriation,180,Irrigation,,,,,,0,,,,,,,,,,http://geohydro.kgs.ku.edu/geohydro/wimas/wate...


## Place of Use Data

In [16]:
#N/A

## Concatenate POD and POU Data.  Make needed changes

In [17]:
# Concatenate dataframes
frames = [outPOD]  # list all out dataframes here
outdf = pd.concat(frames)
outdf = outdf.drop_duplicates().reset_index(drop=True).replace(np.nan, "")
print(len(outdf))

48854


## Clean Data / data types

In [18]:
# Clean name entries of spcial characters
def removeSpecialCharsFunc(Val):
    Val = str(Val)
    Val = re.sub("[$@&.;/\)(-]", "", Val).title().replace("  ", " ").strip().rstrip(',')
    return Val

<>:4: SyntaxWarning: invalid escape sequence '\)'
<>:4: SyntaxWarning: invalid escape sequence '\)'
C:\Users\rjame\AppData\Local\Temp\ipykernel_5460\1085762661.py:4: SyntaxWarning: invalid escape sequence '\)'
  Val = re.sub("[$@&.;/\)(-]", "", Val).title().replace("  ", " ").strip().rstrip(',')


In [19]:
outdf['in_WaterSourceName'] = outdf.apply(lambda row: removeSpecialCharsFunc(row['in_WaterSourceName']), axis=1)
outdf['in_WaterSourceName'].unique()

array(['Sappa Creek', 'Arkansas River', 'S F Solomon River',
       'Bluff Creek Chikaskia', 'N F Ninnescah River', 'Walnut Creek',
       'Whitewoman Creek', 'S F Ninnescah River', 'N F Cimarron River',
       'Little Arkansas River', 'Hackberry Creek', 'Solomon River',
       'Rattlesnake Creek', 'Missouri River', 'Republican River',
       'Chikaskia River', 'Bear Creek', 'Ladder Creek',
       'Smoky Hill River', 'N F Solomon River', 'Kansas River',
       'Big Blue River', 'Crooked Creek', 'Cimarron River',
       'Walnut River', 'Saline River', 'Medicine Lodge River',
       'Beaver Creek', 'Pawnee River', 'Prairie Dog Creek',
       'S F Republican River', 'Cow Creek', 'Big Creek', 'Buckner Creek',
       'Spring River', 'Ninnescah River', 'Sandy Creek',
       'Cottonwood River', 'Delaware River', 'Stranger Creek',
       'Little Blue River', 'Salt Creek', 'S F Big Nemaha River',
       'Marais Des Cygnes River', 'N F Smoky Hill River',
       'Bluff Creek Cimarron', 'Wakarusa 

In [20]:
outdf['in_County'] = outdf.apply(lambda row: removeSpecialCharsFunc(row['in_County']), axis=1)
outdf['in_County'].unique()

array(['Rawlins', 'Haskell', 'Graham', 'Harper', 'Reno', 'Pawnee', 'Lane',
       'Scott', 'Wichita', 'Kingman', 'Stanton', 'Grant', 'Kearny',
       'Harvey', 'Edwards', 'Logan', 'Mitchell', 'Sumner', 'Thomas',
       'Brown', 'Republic', 'Sedgwick', 'Hamilton', 'Ford', 'Trego',
       'Decatur', 'Pottawatomie', 'Gray', 'Pratt', 'Seward', 'Morris',
       'Cowley', 'Morton', 'Saline', 'Butler', 'Finney', 'Geary',
       'Sherman', 'Smith', 'Clay', 'Phillips', 'Wabaunsee', 'Wallace',
       'Ellis', 'Sheridan', 'Mcpherson', 'Rooks', 'Cheyenne', 'Meade',
       'Shawnee', 'Jefferson', 'Stevens', 'Rice', 'Kiowa', 'Gove', 'Rush',
       'Cloud', 'Barton', 'Crawford', 'Greeley', 'Stafford', 'Norton',
       'Ness', 'Russell', 'Douglas', 'Osborne', 'Barber', 'Ottawa',
       'Marion', 'Marshall', 'Washington', 'Lincoln', 'Riley', 'Hodgeman',
       'Cherokee', 'Coffey', 'Ellsworth', 'Dickinson', 'Leavenworth',
       'Comanche', 'Chase', 'Wyandotte', 'Johnson', 'Allen', 'Clark',
       'Jew

In [21]:
outdf['in_SiteName'] = outdf.apply(lambda row: removeSpecialCharsFunc(row['in_SiteName']), axis=1)
outdf['in_SiteName'].unique()

array(['Wade Unspecified'], dtype=object)

In [22]:
outdf['in_AllocationOwner'] = outdf.apply(lambda row: removeSpecialCharsFunc(row['in_AllocationOwner']), axis=1)
outdf['in_AllocationOwner'].unique()

array(['Wade Unspecified'], dtype=object)

In [23]:
# Ensure Empty String / remove string value of "nan"

def ensureEmptyString(val):
    val = str(val).strip()
    if val == "" or val == " " or val == "nan" or pd.isnull(val):
        outString = ""
    else:
        outString = val
    return outString

In [24]:
outdf['in_WaterSourceName'] = outdf.apply(lambda row: ensureEmptyString(row['in_WaterSourceName']), axis=1)
outdf['in_WaterSourceName'].unique()

array(['Sappa Creek', 'Arkansas River', 'S F Solomon River',
       'Bluff Creek Chikaskia', 'N F Ninnescah River', 'Walnut Creek',
       'Whitewoman Creek', 'S F Ninnescah River', 'N F Cimarron River',
       'Little Arkansas River', 'Hackberry Creek', 'Solomon River',
       'Rattlesnake Creek', 'Missouri River', 'Republican River',
       'Chikaskia River', 'Bear Creek', 'Ladder Creek',
       'Smoky Hill River', 'N F Solomon River', 'Kansas River',
       'Big Blue River', 'Crooked Creek', 'Cimarron River',
       'Walnut River', 'Saline River', 'Medicine Lodge River',
       'Beaver Creek', 'Pawnee River', 'Prairie Dog Creek',
       'S F Republican River', 'Cow Creek', 'Big Creek', 'Buckner Creek',
       'Spring River', 'Ninnescah River', 'Sandy Creek',
       'Cottonwood River', 'Delaware River', 'Stranger Creek',
       'Little Blue River', 'Salt Creek', 'S F Big Nemaha River',
       'Marais Des Cygnes River', 'N F Smoky Hill River',
       'Bluff Creek Cimarron', 'Wakarusa 

In [25]:
outdf['in_WaterSourceTypeCV'] = outdf.apply(lambda row: ensureEmptyString(row['in_WaterSourceTypeCV']), axis=1)
outdf['in_WaterSourceTypeCV'].unique()

array(['Groundwater', 'Surface Water'], dtype=object)

In [26]:
outdf['in_SiteTypeCV'] = outdf.apply(lambda row: ensureEmptyString(row['in_SiteTypeCV']), axis=1)
outdf['in_SiteTypeCV'].unique()

array(['WaDE Unspecified'], dtype=object)

In [27]:
outdf['in_SiteName'] = outdf.apply(lambda row: ensureEmptyString(row['in_SiteName']), axis=1)
outdf['in_SiteName'].unique()

array(['Wade Unspecified'], dtype=object)

In [28]:
outdf['in_AllocationOwner'] = outdf.apply(lambda row: ensureEmptyString(row['in_AllocationOwner']), axis=1)
outdf['in_AllocationOwner'].unique()

array(['Wade Unspecified'], dtype=object)

In [29]:
outdf['in_BeneficialUseCategory'] = outdf.apply(lambda row: ensureEmptyString(row['in_BeneficialUseCategory']), axis=1)
uniqueList = list(set([i.strip() for i in ','.join(outdf['in_BeneficialUseCategory'].astype(str)).split(',')]))
uniqueList.sort()
uniqueList

['Domestic',
 'Industrial',
 'Irrigation',
 'Municipal',
 'Recreation',
 'Stockwater']

In [30]:
# Ensure Latitude entry is either numireic or a 0
outdf['in_Latitude'] = pd.to_numeric(outdf['in_Latitude'], errors='coerce').replace(0,"").fillna("")
outdf['in_Latitude'].unique()

array([39.66212, 37.71151, 39.348  , ..., 39.74076, 39.83168, 39.61374],
      shape=(35792,))

In [31]:
# Ensure Longitude entry is either numireic or a 0
outdf['in_Longitude'] = pd.to_numeric(outdf['in_Longitude'], errors='coerce').replace(0,"").fillna("")
outdf['in_Longitude'].unique()

array([-100.99873, -101.07568,  -99.91405, ...,  -96.68485,  -97.82118,
        -98.52036], shape=(34762,))

In [32]:
# Changing datatype of Priority Date to date fields entry
outdf['in_AllocationPriorityDate'] = pd.to_datetime(outdf['in_AllocationPriorityDate'], errors = 'coerce')
outdf['in_AllocationPriorityDate'] = pd.to_datetime(outdf["in_AllocationPriorityDate"].dt.strftime('%m/%d/%Y'))
outdf['in_AllocationPriorityDate'].unique()

C:\Users\rjame\AppData\Local\Temp\ipykernel_5460\1925566652.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  outdf['in_AllocationPriorityDate'] = pd.to_datetime(outdf['in_AllocationPriorityDate'], errors = 'coerce')


<DatetimeArray>
['1975-07-28 00:00:00', '1980-04-15 00:00:00', '1981-11-25 00:00:00',
 '2020-05-11 00:00:00', '1996-08-29 00:00:00', '1973-11-21 00:00:00',
 '1965-01-28 00:00:00', '1945-06-28 00:00:00', '1989-02-24 00:00:00',
 '1965-04-23 00:00:00',
 ...
 '2023-04-25 00:00:00', '2009-03-19 00:00:00', '2020-06-08 00:00:00',
 '2013-11-21 00:00:00', '2014-05-28 00:00:00', '1948-01-09 00:00:00',
 '2024-10-22 00:00:00', '2017-07-24 00:00:00', '1992-09-29 00:00:00',
 '2015-09-25 00:00:00']
Length: 11610, dtype: datetime64[ns]

In [33]:
# Ensure Flow entry is either numireic or a 0
outdf['in_AllocationFlow_CFS'] = pd.to_numeric(outdf['in_AllocationFlow_CFS'], errors='coerce').round(2).replace(0,"").fillna("")
outdf['in_AllocationFlow_CFS'].unique()

array([''], dtype=object)

In [34]:
# Ensure Volume entry is either numireic or a 0
outdf['in_AllocationVolume_AF'] = pd.to_numeric(outdf['in_AllocationVolume_AF'], errors='coerce').round(2).replace(0,"").fillna("")
outdf['in_AllocationVolume_AF'].unique()

array([203.0, 260.0, 59.44, ..., 13.25, 580.25, 575.67],
      shape=(4960,), dtype=object)

In [35]:
# Ensure Irrigated Acreage entry is either numireic or a 0
outdf['in_IrrigatedAcreage'] = pd.to_numeric(outdf['in_IrrigatedAcreage'], errors='coerce').round(2).replace(0,"").fillna("")
outdf['in_IrrigatedAcreage'].unique()

array([''], dtype=object)

In [36]:
# Creating WaDE Custom water source native ID for easy water source identification
# use unique WaterSourceName and WaterSourceType values
# ----------------------------------------------------------------------------------------------------

# Create temp in_WaterSourceNativeID dataframe of unique water source.
def assignIdValueFunc(colRowValue):
    string1 = str(colRowValue)
    outstring = "wadeId" + string1
    return outstring

dfTempID = pd.DataFrame()
dfTempID['in_WaterSourceName'] = outdf['in_WaterSourceName'].astype(str).str.strip()
dfTempID['in_WaterSourceTypeCV'] = outdf['in_WaterSourceTypeCV'].astype(str).str.strip()
dfTempID = dfTempID.drop_duplicates()

dfTempCount = pd.DataFrame(index=dfTempID.index)
dfTempCount["Count"] = range(1, len(dfTempCount.index) + 1)
dfTempID['in_WaterSourceNativeID'] = dfTempCount.apply(lambda row: assignIdValueFunc(row['Count']), axis=1)
dfTempID['linkKey'] = dfTempID['in_WaterSourceName'].astype(str) + dfTempID['in_WaterSourceTypeCV'].astype(str)
IdDict = pd.Series(dfTempID.in_WaterSourceNativeID.values, index=dfTempID.linkKey.astype(str)).to_dict()
# ----------------------------------------------------------------------------------------------------

# Retreive WaDE Custom site native ID
def retrieveIdValueFunc(checkVal, valA, valB):
    checkVal = str(checkVal).strip()
    if checkVal == "":
        linkKeyVal = str(valA).strip() + str(valB).strip()
        outString = IdDict[linkKeyVal]
    else:
        outString = checkVal
    return outString

outdf['in_WaterSourceNativeID'] = outdf.apply(lambda row: retrieveIdValueFunc(row['in_WaterSourceNativeID'], 
                                                                              row['in_WaterSourceName'], row['in_WaterSourceTypeCV']), axis=1)
outdf['in_WaterSourceNativeID'].unique()

array(['wadeId1', 'wadeId2', 'wadeId3', 'wadeId4', 'wadeId5', 'wadeId6',
       'wadeId7', 'wadeId8', 'wadeId9', 'wadeId10', 'wadeId11',
       'wadeId12', 'wadeId13', 'wadeId14', 'wadeId15', 'wadeId16',
       'wadeId17', 'wadeId18', 'wadeId19', 'wadeId20', 'wadeId21',
       'wadeId22', 'wadeId23', 'wadeId24', 'wadeId25', 'wadeId26',
       'wadeId27', 'wadeId28', 'wadeId29', 'wadeId30', 'wadeId31',
       'wadeId32', 'wadeId33', 'wadeId34', 'wadeId35', 'wadeId36',
       'wadeId37', 'wadeId38', 'wadeId39', 'wadeId40', 'wadeId41',
       'wadeId42', 'wadeId43', 'wadeId44', 'wadeId45', 'wadeId46',
       'wadeId47', 'wadeId48', 'wadeId49', 'wadeId50', 'wadeId51',
       'wadeId52', 'wadeId53', 'wadeId54', 'wadeId55', 'wadeId56',
       'wadeId57', 'wadeId58', 'wadeId59', 'wadeId60', 'wadeId61',
       'wadeId62', 'wadeId63', 'wadeId64', 'wadeId65', 'wadeId66',
       'wadeId67', 'wadeId68', 'wadeId69', 'wadeId70', 'wadeId71',
       'wadeId72', 'wadeId73', 'wadeId74', 'wadeId75', 'wad

In [37]:
# Creating WaDE Custom site native ID for easy site identification
# use Unique Latitude, Longitude, SiteName and SiteTypeCV values
# ----------------------------------------------------------------------------------------------------

# Create temp in_SiteNativeID dataframe of unique water source.
def assignIdValueFunc(colRowValue):
    string1 = str(colRowValue)
    outstring = "wadeId" + string1
    return outstring

dfTempID = pd.DataFrame()
dfTempID['in_Latitude'] = outdf['in_Latitude'].astype(str).str.strip()
dfTempID['in_Longitude'] = outdf['in_Longitude'].astype(str).str.strip()
dfTempID['in_SiteName'] = outdf['in_SiteName'].astype(str).str.strip()
dfTempID['in_SiteTypeCV'] = outdf['in_SiteTypeCV'].astype(str).str.strip()
dfTempID = dfTempID.drop_duplicates()

dfTempCount = pd.DataFrame(index=dfTempID.index)
dfTempCount["Count"] = range(1, len(dfTempCount.index) + 1)
dfTempID['in_SiteNativeID'] = dfTempCount.apply(lambda row: assignIdValueFunc(row['Count']), axis=1)
dfTempID['linkKey'] = dfTempID['in_Latitude'].astype(str) + dfTempID['in_Longitude'].astype(str) + dfTempID['in_SiteName'].astype(str)+ dfTempID['in_SiteTypeCV'].astype(str)
IdDict = pd.Series(dfTempID.in_SiteNativeID.values, index=dfTempID.linkKey.astype(str)).to_dict()
# ----------------------------------------------------------------------------------------------------

# Retreive WaDE Custom site native ID
def retrieveIdValueFunc(checkVal, valA, valB, valC, valD):
    checkVal = str(checkVal).strip()
    if checkVal == "":
        linkKeyVal = str(valA).strip() + str(valB).strip() + str(valC).strip() + str(valD).strip()
        outString = IdDict[linkKeyVal]
    else:
        outString = checkVal
    return outString

outdf['in_SiteNativeID'] = outdf.apply(lambda row: retrieveIdValueFunc(row['in_SiteNativeID'], 
                                                                       row['in_Latitude'], row['in_Longitude'],
                                                                       row['in_SiteName'], row['in_SiteTypeCV']), axis=1)
outdf['in_SiteNativeID'].unique()

array(['POD128', 'POD749', 'POD753', ..., 'POD86606', 'POD87979',
       'POD10706'], shape=(39939,), dtype=object)

## Drop non-Active AllocationLegalStatusCV Water Rights
- For this {state name / organization}, we don't want water rights that are considered: {enter string entries here}

In [38]:
# drop non-active AllocationLegalStatusCV values specific to that state.

# drop the list
dropLegalStatusList = [""] # enter string entries here

# drop rows from above list
outdf = outdf[outdf.in_AllocationLegalStatusCV.isin(dropLegalStatusList) == False].reset_index(drop=True)

print(len(outdf))
outdf['in_AllocationLegalStatusCV'].unique()

48813


array(['Certificated Issued', 'Extended Time to Complete',
       'Vested Active', 'Dismissed Pending Completion',
       'Completed Pending Inspection',
       'Inspected Pending Perfection Extended Time to Perfect',
       'Inspected Pending Perfection', 'Approved Pending Completion',
       'Proposed Certificate', 'Dismissed After Certificated Issued',
       'Completed Extended Time to Perfect',
       'Dismissed Pending Perfection', 'Dismissed Pending Inspection',
       'Proposed Certificate Extended Time to Perfect',
       'Dismissed Prior to Approval', 'Dismissed After Vested',
       'Partial Inspection Extended Time to Perfect',
       'Reinstated After Certificate Issued',
       'Completed Partial inspection', 'Reinstated After Vested'],
      dtype=object)

## Shapefile Data
- For attaching geometry to POU csv inputs.

In [39]:
# no POU data at this time

In [40]:
# # # Input File / or use same input as above

# gdfin1 = outdf.copy()
# gdfin1 = gpd.GeoDataFrame(gdfin1, geometry=gdfin1['in_Geometry'], crs="EPSG:4326") # covert to geodataframe
# print(len(gdfin1))
# gdfin1.head()

In [41]:
# # plot shape info to map
# gdfin1.plot()

In [42]:
# # create output for Regulatory Area #1 dataframe
# df = pd.DataFrame()

# columnsList = ['in_SiteNativeID', 'geometry']
# goutdf1 = pd.DataFrame(columns=columnsList, index=gdfin1.index)

# goutdf1['in_SiteNativeID'] =  gdfin1["in_SiteNativeID"].astype(str)  #in_ReportingUnitNativeID needs to match source from above equivalent dataframe
# goutdf1['geometry'] = gdfin1['in_Geometry']
# goutdf1 = goutdf1.drop_duplicates().reset_index(drop=True)

# # drop geometery from outdf
# outdf = outdf.drop(['in_Geometry'], axis=1)


# print(len(goutdf1))
# goutdf1.head()

## Export Data

In [43]:
outdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48813 entries, 0 to 48812
Data columns (total 63 columns):
 #   Column                                        Non-Null Count  Dtype         
---  ------                                        --------------  -----         
 0   WaDEUUID                                      48813 non-null  object        
 1   in_MethodUUID                                 48813 non-null  object        
 2   in_VariableSpecificUUID                       48813 non-null  object        
 3   in_OrganizationUUID                           48813 non-null  object        
 4   in_Geometry                                   48813 non-null  object        
 5   in_GNISFeatureNameCV                          48813 non-null  object        
 6   in_WaterQualityIndicatorCV                    48813 non-null  object        
 7   in_WaterSourceName                            48813 non-null  object        
 8   in_WaterSourceNativeID                        48813 non-null  obje

In [44]:
outdf

,WaDEUUID,in_MethodUUID,in_VariableSpecificUUID,in_OrganizationUUID,in_Geometry,in_GNISFeatureNameCV,in_WaterQualityIndicatorCV,in_WaterSourceName,in_WaterSourceNativeID,in_WaterSourceTypeCV,in_CoordinateAccuracy,in_CoordinateMethodCV,in_County,in_EPSGCodeCV,in_GNISCodeCV,in_HUC12,in_HUC8,in_Latitude,in_Longitude,in_NHDNetworkStatusCV,in_NHDProductCV,in_PODorPOUSite,in_SiteName,in_SiteNativeID,in_SitePoint,in_SiteTypeCV,in_StateCV,in_USGSSiteID,in_AllocationApplicationDate,in_AllocationAssociatedConsumptiveUseSiteIDs,in_AllocationAssociatedWithdrawalSiteIDs,in_AllocationBasisCV,in_AllocationChangeApplicationIndicator,in_AllocationCommunityWaterSupplySystem,in_AllocationCropDutyAmount,in_AllocationExpirationDate,in_AllocationFlow_CFS,in_AllocationLegalStatusCV,in_AllocationNativeID,in_AllocationOwner,in_AllocationPriorityDate,in_AllocationSDWISIdentifierCV,in_AllocationTimeframeEnd,in_AllocationTimeframeStart,in_AllocationTypeCV,in_AllocationVolume_AF,in_BeneficialUseCategory,in_CommunityWaterSupplySystem,in_CropTypeCV,in_CustomerTypeCV,in_DataPublicationDate,in_DataPublicationDOI,in_ExemptOfVolumeFlowPriority,in_GeneratedPowerCapacityMW,in_IrrigatedAcreage,in_IrrigationMethodCV,in_LegacyAllocationIDs,in_OwnerClassificationCV,in_PopulationServed,in_PowerType,in_PrimaryBeneficialUseCategory,in_SDWISIdentifierCV,in_WaterAllocationNativeURL
0,d35260,KSwr_M1,KSwr_V1,KSwr_O1,,,,Sappa Creek,wadeId1,Groundwater,WaDE Unspecified,WaDE Unspecified,Rawlins,4326,,,,39.66212,-100.99873,,,POD,Wade Unspecified,POD128,,WaDE Unspecified,KS,,,,,,,,,,,Certificated Issued,ks24738,Wade Unspecified,1975-07-28,,,,Appropriation,203.00000,Irrigation,,,,,,0,,,,,,,,,,http://geohydro.kgs.ku.edu/geohydro/wimas/wate...
1,d52784,KSwr_M1,KSwr_V1,KSwr_O1,,,,Arkansas River,wadeId2,Groundwater,WaDE Unspecified,WaDE Unspecified,Haskell,4326,,,,37.71151,-101.07568,,,POD,Wade Unspecified,POD749,,WaDE Unspecified,KS,,,,,,,,,,,Certificated Issued,ks34638,Wade Unspecified,1980-04-15,,,,Appropriation,260.00000,Irrigation,,,,,,0,,,,,,,,,,http://geohydro.kgs.ku.edu/geohydro/wimas/wate...
2,d57232,KSwr_M1,KSwr_V1,KSwr_O1,,,,S F Solomon River,wadeId3,Groundwater,WaDE Unspecified,WaDE Unspecified,Graham,4326,,,,39.34800,-99.91405,,,POD,Wade Unspecified,POD753,,WaDE Unspecified,KS,,,,,,,,,,,Certificated Issued,ks36303,Wade Unspecified,1981-11-25,,,,Appropriation,59.44000,Irrigation,,,,,,0,,,,,,,,,,http://geohydro.kgs.ku.edu/geohydro/wimas/wate...
3,d86128,KSwr_M1,KSwr_V1,KSwr_O1,,,,Bluff Creek Chikaskia,wadeId4,Groundwater,WaDE Unspecified,WaDE Unspecified,Harper,4326,,,,37.36612,-98.17381,,,POD,Wade Unspecified,POD89686,,WaDE Unspecified,KS,,,,,,,,,,,Extended Time to Complete,ks86945,Wade Unspecified,2020-05-11,,,,Appropriation,224.00000,Irrigation,,,,,,0,,,,,,,,,,http://geohydro.kgs.ku.edu/geohydro/wimas/wate...
4,d63061,KSwr_M1,KSwr_V1,KSwr_O1,,,,N F Ninnescah River,wadeId5,Groundwater,WaDE Unspecified,WaDE Unspecified,Reno,4326,,,,37.79939,-97.99142,,,POD,Wade Unspecified,POD1676,,WaDE Unspecified,KS,,,,,,,,,,,Certificated Issued,ks43091,Wade Unspecified,1996-08-29,,,,Appropriation,180.00000,Irrigation,,,,,,0,,,,,,,,,,http://geohydro.kgs.ku.edu/geohydro/wimas/wate...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48808,d79625,KSwr_M1,KSwr_V1,KSwr_O1,,,,Neosho River,wadeId65,Surface Water,WaDE Unspecified,WaDE Unspecified,Neosho,4326,,,,37.47418,-95.15695,,,POD,Wade Unspecified,POD76552,,WaDE Unspecified,KS,,,,,,,,,,,Certificated Issued,ks71697,Wade Unspecified,2009-05-14,,,,Appropriation,73.00000,Recreation,,,,,,0,,,,,,,,,,http://geohydro.kgs.ku.edu/geohydro/wimas/wate...
48809,d9017,KSwr_M1,KSwr_V1,KSwr_O1,,,,Prairie Dog Creek,wadeId56,Surface Water,WaDE Unspecified,WaDE Unspecified,Norton,4326,,,,39.77369,-100.08860,,,POD,Wade Unspecified,POD407,,WaDE Unspecified,

In [45]:
# Export the output dataframe
outdf.to_csv('RawInputData/Pwr_Main.zip', compression=dict(method='zip', archive_name='Pwr_Main.csv'), index=False)  # The output, save as a zip
#goutdf1.to_csv('RawInputData/P_Geometry.zip', compression=dict(method='zip', archive_name='P_Geometry.csv'), index=False)  # The output geometry.